# Pistachio certificate premium to weekly Dahan-Bast prices

This is an indicative comparison of traded `PistaCL` settlement prices with the latest available Abtahi Dahan-Bast weekly midpoint. It does not alter or filter the source workbook or the existing audit. The weekly quote unit is not stated in the workbook; the calculation below assumes **toman per kilogram**, converts it to rial per kilogram at 10 rial per toman, and should be revisited if Abtahi confirms a different unit.

Each IME certificate represents one kilogram of pistachio. The published contract is Fandoqi Dahan-Bast, size 30-32; the Abtahi workbook does not state an equivalent grade or size. [IME announcement](https://tg.me/boursekalairan/18516). This is a weekly proxy comparison, not an approved physical-market benchmark.

In [ ]:
from pathlib import Path

import jdatetime
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        for candidate in (base, base / 'goods' / 'pista'):
            if (candidate / 'data/raw/physical/Pistachio_Weekly_Prices.xlsx').exists():
                return candidate
    raise FileNotFoundError('Could not locate the Pista project source workbook')

PROJECT_ROOT = find_project_root()
PHYSICAL_PRICE_UNIT = 'toman_per_kg'  # Provisional; confirm with Abtahi.
MAX_PHYSICAL_AGE_DAYS = 6
if PHYSICAL_PRICE_UNIT != 'toman_per_kg':
    raise ValueError('Set a verified physical-to-IRR conversion before running')
RIAL_PER_TOMAN = 10
PHYSICAL_PATH = PROJECT_ROOT / 'data/raw/physical/Pistachio_Weekly_Prices.xlsx'
CERTIFICATE_PATH = PROJECT_ROOT / 'data/raw/certificate/pista_certificate_raw.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data/processed/bubble/pista_certificate_bubble.csv'

## Source observations and alignment

Dahan-Bast price is recomputed as `(Min + Max) / 2`. A certificate day qualifies only when `TradesVolume > 0` and `TodaySettlementPrice > 0`. The latest physical observation dated on or before that day is used only if it is at most six calendar days old. No future price, interpolation, or unbounded fill is used. Exact calendar-date overlaps in this extract have zero certificate volume, so they are excluded from the traded-day comparison.

In [ ]:
def jalali_to_gregorian(value: str):
    try:
        return jdatetime.date(*map(int, value.split('/'))).togregorian()
    except (TypeError, ValueError, AttributeError):
        return None

physical = pd.read_excel(PHYSICAL_PATH, engine='openpyxl')
required_physical = {'Date (Jalali)', 'Dahan-Bast Min Price', 'Dahan-Bast Max Price'}
if not required_physical.issubset(physical.columns):
    raise ValueError('Expected Dahan-Bast source columns are missing')
physical['physical_date'] = pd.to_datetime(
    physical['Date (Jalali)'].map(jalali_to_gregorian), errors='coerce'
).astype('datetime64[ns]')
physical['physical_min_toman_per_kg'] = pd.to_numeric(physical['Dahan-Bast Min Price'], errors='coerce')
physical['physical_max_toman_per_kg'] = pd.to_numeric(physical['Dahan-Bast Max Price'], errors='coerce')
physical['physical_mid_toman_per_kg'] = (
    physical['physical_min_toman_per_kg'] + physical['physical_max_toman_per_kg']
) / 2
physical = physical.dropna(subset=['physical_date', 'physical_mid_toman_per_kg']).copy()
physical = physical.loc[
    (physical['physical_min_toman_per_kg'] > 0)
    & (physical['physical_min_toman_per_kg'] <= physical['physical_max_toman_per_kg'])
].sort_values('physical_date')
if physical['physical_date'].duplicated().any():
    raise ValueError('Duplicate usable physical dates need manual resolution')

certificate = pd.read_csv(CERTIFICATE_PATH, encoding='utf-8-sig')
required_certificate = {'DT', 'PersianDate', 'ContractCode', 'TradesVolume', 'TodaySettlementPrice'}
if not required_certificate.issubset(certificate.columns):
    raise ValueError('Expected certificate source columns are missing')
certificate = certificate.loc[certificate['ContractCode'].eq('PistaCL')].copy()
certificate['certificate_date'] = pd.to_datetime(certificate['DT'].str[:10]).astype('datetime64[ns]')
certificate['TradesVolume'] = pd.to_numeric(certificate['TradesVolume'], errors='raise')
certificate['TodaySettlementPrice'] = pd.to_numeric(certificate['TodaySettlementPrice'], errors='raise')
certificate = certificate.loc[
    (certificate['TradesVolume'] > 0) & (certificate['TodaySettlementPrice'] > 0)
].sort_values('certificate_date')
if certificate['certificate_date'].duplicated().any():
    raise ValueError('Duplicate traded certificate dates need manual resolution')
print(f'Positive-volume certificate dates: {len(certificate)}')
print(f'Usable Dahan-Bast weekly observations: {len(physical)}')

In [ ]:
physical_for_join = physical[[
    'physical_date', 'Date (Jalali)',
    'physical_min_toman_per_kg', 'physical_max_toman_per_kg', 'physical_mid_toman_per_kg'
]].rename(columns={'Date (Jalali)': 'physical_date_jalali'})
matched = pd.merge_asof(
    certificate, physical_for_join,
    left_on='certificate_date', right_on='physical_date',
    direction='backward', tolerance=pd.Timedelta(days=MAX_PHYSICAL_AGE_DAYS),
)
unmatched = int(matched['physical_date'].isna().sum())
bubble = matched.dropna(subset=['physical_date']).copy()
bubble['physical_age_days'] = (bubble['certificate_date'] - bubble['physical_date']).dt.days
bubble['physical_mid_irr_per_kg'] = bubble['physical_mid_toman_per_kg'] * RIAL_PER_TOMAN
bubble['certificate_irr_per_kg'] = bubble['TodaySettlementPrice']
bubble['bubble_pct'] = 100 * (
    bubble['certificate_irr_per_kg'] / bubble['physical_mid_irr_per_kg'] - 1
)
bubble['spread_irr_per_kg'] = bubble['certificate_irr_per_kg'] - bubble['physical_mid_irr_per_kg']
bubble['physical_unit_assumption'] = PHYSICAL_PRICE_UNIT
bubble = bubble.rename(columns={
    'PersianDate': 'certificate_date_jalali',
    'TradesVolume': 'certificate_trades_volume',
})[[
    'certificate_date', 'certificate_date_jalali', 'physical_date',
    'physical_date_jalali', 'physical_age_days', 'certificate_trades_volume',
    'certificate_irr_per_kg', 'physical_min_toman_per_kg',
    'physical_max_toman_per_kg', 'physical_mid_toman_per_kg',
    'physical_mid_irr_per_kg', 'spread_irr_per_kg', 'bubble_pct',
    'physical_unit_assumption',
]]
if bubble.empty:
    raise ValueError('No traded certificate dates have a Dahan-Bast observation within six days')
if not bubble['physical_age_days'].between(0, MAX_PHYSICAL_AGE_DAYS).all():
    raise AssertionError('A physical observation is outside the allowed age window')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
bubble.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'Matched trading days: {len(bubble)}; unmatched trading days: {unmatched}')
display(bubble[['certificate_date_jalali', 'physical_date_jalali',
                'physical_age_days', 'certificate_irr_per_kg',
                'physical_mid_irr_per_kg', 'bubble_pct']].round({'bubble_pct': 2}))

In [ ]:
figure = go.Figure()
figure.add_trace(go.Scatter(
    x=bubble['certificate_date'], y=bubble['bubble_pct'],
    mode='lines+markers', name='Indicative premium',
    marker=dict(size=8, color='#167d8d'),
    line=dict(color='#167d8d', width=2),
    customdata=bubble[['certificate_date_jalali', 'physical_date_jalali',
                       'physical_age_days', 'certificate_trades_volume']].to_numpy(),
    hovertemplate=(
        'Certificate: %{customdata[0]}<br>'
        'Physical quote: %{customdata[1]} (%{customdata[2]} days old)<br>'
        'Volume: %{customdata[3]:,.0f} certificates<br>'
        'Premium: %{y:.2f}%<extra></extra>'
    ),
))
figure.add_hline(y=0, line_color='#59636e', line_width=1)
figure.update_layout(
    title='PistaCL premium to latest Abtahi Dahan-Bast weekly price (provisional)',
    xaxis_title='Certificate trading date', yaxis_title='Premium / discount (%)',
    template='plotly_white', height=480, hovermode='closest',
    margin=dict(l=70, r=30, t=75, b=65),
)
figure.show()

A positive value means the certificate settlement exceeded the matched weekly Dahan-Bast observation after the assumed toman-to-rial conversion. Repeated weekly quotes are deliberately shown with their age. The chart does not establish arbitrage or a validated certificate bubble: the Abtahi quote unit, grade, market basis, and costs still require confirmation.

## Historical bubble distribution

This section reads the standardized processed table and renders an interactive Plotly figure for
each bubble type. The panels show the observed distribution, empirical cumulative distribution
function F(x), and magnitude frequency P(|Bubble| >= |x|). Negative bubbles retain their sign in
the first two panels; the third panel measures magnitude only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def locate_distribution_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "shared").exists() and (candidate / "goods/pista").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

distribution_workspace = locate_distribution_workspace()
if str(distribution_workspace) not in sys.path:
    sys.path.insert(0, str(distribution_workspace))

from shared.market_analysis.bubble_distribution import plot_distribution_plotly

distribution_project = distribution_workspace / "goods/pista"
distribution_files = list(
    (distribution_project / "data/processed/bubble").glob("*_bubble_distribution.csv")
)
if len(distribution_files) != 1:
    raise ValueError(f"Expected one named bubble distribution CSV, found {distribution_files}")
bubble_distribution = pd.read_csv(distribution_files[0], parse_dates=["observation_date"])
for series_id, series_distribution in bubble_distribution.groupby("series_id", sort=True):
    comparison = series_distribution["comparison"].iloc[0]
    figure = plot_distribution_plotly(series_distribution, comparison)
    figure.show()

display(bubble_distribution)